## File Search (파일 검색)

Gemini API는 파일 검색 도구를 통해 검색 증강 생성(RAG)을 활성화합니다. 파일 검색은 데이터를 가져오고, 청크하고, 인덱싱하여 사용자의 프롬프트에 따라 관련 정보를 빠르게 검색할 수 있도록 합니다. 그런 다음 이 정보는 모델에 대한 컨텍스트로 제공되어 모델이 보다 정확하고 관련성 높은 답변을 제공할 수 있도록 합니다.

In [12]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [13]:
# 예시 질의
query = "본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 어떻게 되나요?"
# query = "Advance RAG 기법이 임상시험 데이터 분석에서 수행하는 주요 역할은 무엇인가요?"
# query = "본 연구에서 Private LLM 성능을 평가하기 위해 사용한 지표 3가지는 무엇인가요?"
# query = "국내에서 LLM을 임상시험에 적용한 대표적인 기관과 그 적용 사례를 2가지 이상 말해보세요."
# query = "ROUGE 평가에서 Private LLM과 ChatGPT의 Recall 값은 각각 얼마였나요?"

In [14]:
file_path = "..\\data\\KCI_FI003153549.pdf"

### 파일 검색 저장소에 직접 업로드

In [ ]:
from google import genai
from google.genai import types
import time

client = genai.Client()

# Create the File Search store with an optional display name
file_search_store = client.file_search_stores.create(config={'display_name': 'your-fileSearchStore-name'})

# Upload and import a file into the File Search store, supply a file name which will be visible in citations
operation = client.file_search_stores.upload_to_file_search_store(
  file=file_path,
  file_search_store_name=file_search_store.name,
  config={
      'display_name' : 'display-file-name',
  }
)

# Wait until import is complete
while not operation.done:
    time.sleep(5)
    operation = client.operations.get(operation)

# Ask a question about the file
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=query,
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                file_search=types.FileSearch(
                    file_search_store_names=[file_search_store.name]
                )
            )
        ]
    )
)

print(response.text)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


I am sorry, but I do not have any information about Robert Graves in my current knowledge base.


### 파일 가져오기

In [ ]:
from google import genai
from google.genai import types
import time

client = genai.Client()

# Upload the file using the Files API, supply a file name which will be visible in citations
sample_file = client.files.upload(file=file_path, config={'display_name': 'display_file_name'})

# # Create the File Search store with an optional display name
file_search_store = client.file_search_stores.create(config={'display_name': 'your-fileSearchStore-name'})



Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [19]:
file_search_store.name

'fileSearchStores/yourfilesearchstorename-c4vc1s1m759y'

In [22]:
sample_file.name

'files/ckfx0tm98sdk'

In [ ]:
# Import the file into the File Search store
operation = client.file_search_stores.import_file(
    file_search_store_name=file_search_store.name,
    file_name=sample_file.name
)


In [ ]:
operation

ImportFileOperation(
  name='fileSearchStores/yourfilesearchstorename-c4vc1s1m759y/operations/ckfx0tm98sdk-fskftt4m8d8c',
  response=ImportFileResponse()
)

In [25]:
not operation.done

True

In [26]:
client.operations.get(operation)

ImportFileOperation(
  done=True,
  name='fileSearchStores/yourfilesearchstorename-c4vc1s1m759y/operations/ckfx0tm98sdk-fskftt4m8d8c',
  response=ImportFileResponse(
    document_name='ckfx0tm98sdk-fskftt4m8d8c',
    parent='yourfilesearchstorename-c4vc1s1m759y'
  )
)

In [27]:

# Wait until import is complete
while not operation.done:
    time.sleep(5)
    operation = client.operations.get(operation)


In [28]:
operation

ImportFileOperation(
  done=True,
  name='fileSearchStores/yourfilesearchstorename-c4vc1s1m759y/operations/ckfx0tm98sdk-fskftt4m8d8c',
  response=ImportFileResponse(
    document_name='ckfx0tm98sdk-fskftt4m8d8c',
    parent='yourfilesearchstorename-c4vc1s1m759y'
  )
)

In [29]:
file_search_store.name

'fileSearchStores/yourfilesearchstorename-c4vc1s1m759y'

In [ ]:
# Ask a question about the file
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=query,
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                file_search=types.FileSearch(
                    file_search_store_names=[file_search_store.name]
                )
            )
        ]
    )
)

print(response.text)

In [34]:
# Create a File Search store (including optional display_name for easier reference)
for file_search_store in client.file_search_stores.list():
    print(file_search_store)

In [33]:
# Delete a File Search store
for file_search_store in client.file_search_stores.list():
    client.file_search_stores.delete(name=file_search_store.name, config={'force': True})